# Phase1 Self-Planning 결과 분석

LiveCodeBench v6의 **stdin subset**에서 수행한 Direct Code Generation 결과를 분석한다.

기본 입력 파일: `outputs/self_plan
_stdin/results.jsonl`


In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)


## 1. 결과 파일 로드


In [2]:
RESULT_PATH = Path('../self_plan_stdin/results.jsonl')

if not RESULT_PATH.exists():
    raise FileNotFoundError(
        f'Result file not found: {RESULT_PATH.resolve()}'
    )

records = []
with RESULT_PATH.open('r', encoding='utf-8') as file:
    for line_number, line in enumerate(file, start=1):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as error:
            raise ValueError(f'Invalid JSON at line {line_number}') from error

df = pd.DataFrame(records)
print(f'Loaded records: {len(df)}')
print(f'Columns: {len(df.columns)}')
# display(df.head())


Loaded records: 10
Columns: 25


In [3]:
print(df.columns.tolist())

['problem_id', 'dataset', 'strategy', 'model_name', 'seed', 'title', 'platform', 'contest_id', 'contest_date', 'difficulty', 'problem', 'formatted_prompt', 'raw_output', 'extracted_code', 'prompt_tokens', 'completion_tokens', 'generation_time', 'passed', 'status', 'passed_tests', 'total_tests', 'execution_time', 'error_message', 'test_results', 'strategy_trace']


## 2. 첫 번째 샘플의 strategy_trace 보기


In [4]:
sample = df.iloc[0]

# sample["strategy_trace"]

In [5]:
# plan step만 꺼내기
sample = df.iloc[0]

plan_step = next(
    step
    for step in sample["strategy_trace"]
    if step["name"] == "plan_generation"
)

plan_step

{'name': 'plan_generation',
 'formatted_prompt': 'Analyze the following competitive programming problem and produce a concise implementation plan.\n\nRequirements:\n- Use at most 6 bullet points.\n- Do not repeat or summarize the problem statement.\n- State the core algorithm and exact decision condition.\n- State the key invariant or correctness reasoning.\n- Mention only the necessary data structures.\n- Mention important edge cases.\n- State time and space complexity in one bullet.\n- Do not write code or pseudocode.\n- Do not include headings, introductions, or conclusions.\n- Return only the bullet-point plan.\n\nProblem Title:\nA. Short Sort\n\nProblem:\nThere are three cards with letters $\\texttt{a}$, $\\texttt{b}$, $\\texttt{c}$ placed in a row in some order. You can do the following operation at most once: \n\n \n-  Pick two cards, and swap them.  Is it possible that the row becomes $\\texttt{abc}$ after the operation? Output "YES" if it is possible, and "NO" otherwise.\n\nIn

In [6]:
# print(plan_step["completion_tokens"])
# 출력한 토큰 길이 확인

In [7]:
print(plan_step["raw_output"][-1000:])

print("=" * 80)
print(plan_step["raw_output"][-800:])

esired order $\texttt{abc}$. If not, check if there is exactly one pair of adjacent characters that can be swapped to achieve the sorted order.
- **Decision Condition**: The string is sorted if it is either $\texttt{abc}$ or $\texttt{acb}$ or $\texttt{bac}$.
- **Key Invariant**: The string must have at most one pair of adjacent characters that are out of order for a valid solution.
- **Data Structures**: No additional data structures are needed beyond the input string.
- **Edge Cases**: 
  - If the string is already $\texttt{abc}$, output "YES".
  - If the string is $\texttt{acb}$ or $\texttt{bac}$, output "YES".
  - If the string is $\texttt{cba}$, output "NO".
  - If the string has more than one pair of adjacent characters out of order, output "NO".
- **Time Complexity**: $O(n)$, where $n$ is the length of the string. This is because we only need to traverse the string once to check the conditions.
- **Space Complexity**: $O(1)$, as we are only using a constant amount of extra space.

### 3. 모든 문제에서 확인하기

In [8]:
rows = []

for _, row in df.iterrows():

    plan_step = next(
        step
        for step in row["strategy_trace"]
        if step["name"] == "plan_generation"
    )

    rows.append({
        "problem_id": row["problem_id"],
        "title": row["title"],
        "plan_tokens": plan_step["completion_tokens"],
        "hit_limit": plan_step["completion_tokens"] == 512,
        "plan_tail": plan_step["raw_output"][-120:]
    })

plan_df = pd.DataFrame(rows)

display(plan_df)

,problem_id,title,plan_tokens,hit_limit,plan_tail
0,1873_A,A. Short Sort,271,False,"ing once to check the conditions.\n- **Space Complexity**: $O(1)$, as we are only using a constant amount of extra s..."
1,1873_B,B. Good Kid,245,False,"igits once.\n- **Space Complexity**: O(1), as we are using a constant amount of extra space regardless of the input ..."
2,1873_D,D. 1D Eraser,215,False,", we process each character once.\n- **Space Complexity**: $O(n)$, due to the deque used to store the window of size..."
3,1883_B,B. Chemistry,170,False,cies and checking the condition.\n- **Space Complexity**: $O(1)$ since the character set is fixed (26 lowercase lett...
4,1883_C,C. Raspberries,210,False,"is the length of the array.\n- **Space Complexity**: $O(1)$, as the space used is constant regardless of the input ..."
5,1899_A,A. Game with Integers,269,False,"where $t$ is the number of test cases.\n- **Space Complexity**: $O(1)$, as only a constant amount of extra space is ..."
6,1899_B,B. 250 Thousand Tons of TNT,142,False,ly.\n- **Time Complexity**: $O(t \cdot n \log n)$ due to sorting.\n- **Space Complexity**: $O(n)$ for storing the we...
7,1899_C,C. Yarik and Array,228,False,"ber of times.\n- **Space Complexity**: $O(1)$, as we only use a fixed amount of extra space regardless of the input ..."
8,1899_D,D. Yarik and Musical Notes,225,False,"n all arrays combined.\n- **Space Complexity**: $O(m)$, where $m$ is the total number of elements in all arrays comb..."
9,abc301_a,Overall Winner,248,False,"once to count the wins.\n- **Space Complexity**: O(1), as we only use a constant amount of extra space for the coun..."


In [9]:
for _, row in df.iterrows():
    plan_step = next(
        step
        for step in row["strategy_trace"]
        if step["name"] == "plan_generation"
    )

    print("=" * 100)
    print(
        row["problem_id"],
        "|",
        row["title"],
        "|",
        row["status"],
    )
    print("=" * 100)
    print(plan_step["raw_output"])
    print()

1873_A | A. Short Sort | WRONG_ANSWER
- **Core Algorithm**: Check if the input string is already sorted in the desired order $\texttt{abc}$. If not, check if there is exactly one pair of adjacent characters that can be swapped to achieve the sorted order.
- **Decision Condition**: The string is sorted if it is either $\texttt{abc}$ or $\texttt{acb}$ or $\texttt{bac}$.
- **Key Invariant**: The string must have at most one pair of adjacent characters that are out of order for a valid solution.
- **Data Structures**: No additional data structures are needed beyond the input string.
- **Edge Cases**: 
  - If the string is already $\texttt{abc}$, output "YES".
  - If the string is $\texttt{acb}$ or $\texttt{bac}$, output "YES".
  - If the string is $\texttt{cba}$, output "NO".
  - If the string has more than one pair of adjacent characters out of order, output "NO".
- **Time Complexity**: $O(n)$, where $n$ is the length of the string. This is because we only need to traverse the string once

In [10]:
def show_plan(problem_id: str) -> None:
    row = df.loc[
        df["problem_id"] == problem_id
    ].iloc[0]

    plan_step = next(
        step
        for step in row["strategy_trace"]
        if step["name"] == "plan_generation"
    )

    print("=" * 100)
    print(
        f"{row['problem_id']} | "
        f"{row['title']} | "
        f"{row['difficulty']} | "
        f"{row['status']}"
    )
    print("=" * 100)

    print("\n[Generated Plan]")
    print(plan_step["raw_output"])

    print("\n[Extracted Code]")
    print(row["extracted_code"])

    print("\n[Error]")
    print(row["error_message"])

show_plan("1873_A")

1873_A | A. Short Sort | easy | WRONG_ANSWER

[Generated Plan]
- **Core Algorithm**: Check if the input string is already sorted in the desired order $\texttt{abc}$. If not, check if there is exactly one pair of adjacent characters that can be swapped to achieve the sorted order.
- **Decision Condition**: The string is sorted if it is either $\texttt{abc}$ or $\texttt{acb}$ or $\texttt{bac}$.
- **Key Invariant**: The string must have at most one pair of adjacent characters that are out of order for a valid solution.
- **Data Structures**: No additional data structures are needed beyond the input string.
- **Edge Cases**: 
  - If the string is already $\texttt{abc}$, output "YES".
  - If the string is $\texttt{acb}$ or $\texttt{bac}$, output "YES".
  - If the string is $\texttt{cba}$, output "NO".
  - If the string has more than one pair of adjacent characters out of order, output "NO".
- **Time Complexity**: $O(n)$, where $n$ is the length of the string. This is because we only need to

In [11]:
# 각 문장 끝 확인
for _, row in plan_df.iterrows():

    if row["hit_limit"]:
        print("=" * 80)
        print(row["problem_id"])
        print(row["plan_tail"])
        print()